# Module 1.2 — RAG: Giving the Agent a Library

**Where we are.** In notebook 02 we gave the LLM *computational* tools (calculator, unit converter). Those fixed arithmetic hallucinations, but the model still invents citations, mis-remembers dates, and confuses algorithms. It cannot look anything up.

**What RAG fixes.** **Retrieval-Augmented Generation** plugs a *read-only library* into the agent. At every query we:

1. **embed** the question into a 384-dimensional vector via a a sentence embedding model (i.e. a learned map $\phi : \text{text} \to \mathbb{R}^{384}$)
2. **retrieve** the $K$ paper passages closest to it in that vector space,
3. **inject** those passages into the prompt as context,
4. let the model **generate** an answer grounded in the passages.

The model still produces the prose, but the *facts* are now verbatim from real papers, and we can cite them.

**Plan.** We'll build the six-stage pipeline in plain Python, one cell per stage. Then we wrap the final retrieval step as a `smolagents` tool and drop it into the Module 1.1 agent. The agent goes from "knows arithmetic" to "knows arithmetic *and our arXiv corpus*."


## 1. Load — read PDFs from `data/papers/`

`setup/download_papers.py` already dropped 8 arXiv PDFs on the Ising model into `data/papers/` along with a `metadata.json` file listing titles, authors, and abstracts. Here we convert each PDF into raw text with `pypdf`, keeping the page boundaries so we can cite page numbers later.

> **Note on PDF extraction quality.** PDF text extraction is lossy: equations turn into strings like "T c= 2 /ln(1 +√2)", column layouts interleave awkwardly, and figure captions float to strange places. For a research tool you'd use GROBID or Nougat; for this course the `pypdf` output is good enough, and the fragility is itself pedagogically useful (§9 exercise 2).


In [34]:
import json
from pathlib import Path
from typing import Any

from pypdf import PdfReader

PAPERS_DIR = Path("../data/papers")
assert PAPERS_DIR.exists(), (
    f"Expected paper corpus at {PAPERS_DIR.resolve()}. "
    "Did you run setup/download_papers.py?"
)

# Load metadata.json for titles / arxiv ids
meta_path = PAPERS_DIR / "metadata.json"
metadata = json.loads(meta_path.read_text()) if meta_path.exists() else []
# The file is a list of dicts keyed by arxiv id; build a lookup by PDF filename stem.
meta_by_id: dict[str, dict] = {m["arxiv_id"].replace("/", "_"): m for m in metadata}
print(f"Corpus metadata entries: {len(meta_by_id)}")


def load_paper(pdf_path: Path) -> dict[str, Any]:
    """Return {'id', 'title', 'pages': [page1_text, page2_text, ...]}."""
    stem_id = pdf_path.stem.split("_")[0]
    # Old-style ids look like "cond-mat_0508666" after our filename sanitisation.
    if stem_id in ("cond-mat", "physics", "hep-th", "math-ph"):
        stem_id = f"{stem_id}_{pdf_path.stem.split('_')[1]}"
    meta = meta_by_id.get(stem_id, {})
    reader = PdfReader(str(pdf_path))
    pages = [(p.extract_text() or "").strip() for p in reader.pages]
    return {
        "id":     stem_id,
        "title":  meta.get("title", pdf_path.stem),
        "pages":  pages,
        "source": pdf_path.name,
    }


pdfs = sorted(PAPERS_DIR.glob("*.pdf"))
papers = [load_paper(p) for p in pdfs]
print(f"Loaded {len(papers)} papers:")
for p in papers:
    n_chars = sum(len(pg) for pg in p["pages"])
    print(f"  [{p['id']:25s}]  {len(p['pages']):3d} pages  {n_chars:>7d} chars  —  {p['title'][:60]}")


Corpus metadata entries: 8
Loaded 8 papers:
  [1404.4567                ]    7 pages    27629 chars  —  Rényi Information flow in the Ising model with single-spin d
  [1808.10426               ]    6 pages    24788 chars  —  Phase ordering kinetics of the long-range Ising model
  [2008.00245               ]   10 pages    32597 chars  —  Two temperature Ising Model
  [2411.06819               ]   16 pages    42659 chars  —  Monte Carlo Simulation of Anisotropic Ising Model Using Metr
  [cond-mat_0508666         ]   13 pages    12328 chars  —  Mixed algorithms in the Ising model on directed Barabasi-Alb
  [cond-mat_0508719         ]    4 pages     9606 chars  —  Static and dynamic critical behaviour of 3d random site Isin
  [cond-mat_0603521         ]   24 pages    53170 chars  —  Local and cluster critical dynamics of the 3d random-site Is
  [cond-mat_0607220         ]    8 pages    10580 chars  —  Ising model spin S=1 on directed Barabasi-Albert networks


## 2. Chunk — split into overlapping passages

Embedding a full 16-page paper into a single 384-dim vector would average everything out and destroy the signal. We split each paper into small passages (~400 words) with a small overlap (~50 words) so that a concept that straddles a boundary isn't cut in half.

Chunking strategy is a real design axis:

| Strategy | What it does | Trade-off |
|---|---|---|
| Fixed size (what we use) | N words per chunk, with overlap | simple, cheap, topic-ignorant |
| Semantic | Split at paragraph / section breaks | nicer boundaries, heavier parser |
| Hierarchical | Keep a per-section summary + fine-grained chunks | two-stage retrieval, more complexity |
| Embedding-based | Merge adjacent chunks with similar embeddings | adaptive, expensive upfront |

For the 2D-Ising corpus, fixed-size 400/50 works fine. For legal contracts or code, it doesn't. This is the kind of knob where benchmarking beats intuition — but we have to pick *some* knob.


In [35]:
def chunk_paper(paper: dict, size: int = 400, overlap: int = 50) -> list[dict]:
    """Return a list of {'doc_id', 'source', 'title', 'page_start', 'text'}.

    We glue the full paper text together with a sentinel marker between pages
    so we can recover (approximately) which page each chunk started on.
    """
    assert overlap < size, "overlap must be smaller than chunk size"
    # Annotate pages with their index, then join.
    marked_pages = [(i + 1, text) for i, text in enumerate(paper["pages"]) if text]
    # Build a flat word stream with a parallel page-number stream.
    words: list[str] = []
    word_page: list[int] = []
    for pg_idx, pg_text in marked_pages:
        pg_words = pg_text.split()
        words.extend(pg_words)
        word_page.extend([pg_idx] * len(pg_words))

    chunks: list[dict] = []
    step = size - overlap
    for start in range(0, len(words), step):
        end = start + size
        if start >= len(words):
            break
        chunk_text = " ".join(words[start:end])
        if len(chunk_text.split()) < 30:   # drop tiny tail chunks
            continue
        chunks.append({
            "doc_id":     f"{paper['id']}__c{len(chunks):03d}",
            "arxiv_id":   paper["id"],
            "source":     paper["source"],
            "title":      paper["title"],
            "page_start": word_page[start] if start < len(word_page) else -1,
            "text":       chunk_text,
        })
    return chunks


all_chunks: list[dict] = []
for p in papers:
    all_chunks.extend(chunk_paper(p))

print(f"Total chunks: {len(all_chunks)}")
print(f"Avg words/chunk: {sum(len(c['text'].split()) for c in all_chunks) / len(all_chunks):.0f}")
print("\nExample chunk:")
print(f"  id:    {all_chunks[0]['doc_id']}")
print(f"  page:  {all_chunks[0]['page_start']}")
print(f"  text:  {all_chunks[0]['text'][:200]}...")


Total chunks: 97
Avg words/chunk: 385

Example chunk:
  id:    1404.4567__c000
  page:  1
  text:  arXiv:1404.4567v2 [cond-mat.stat-mech] 2 Nov 2014R´ enyi Information ﬂow in the Ising model with single-spin d ynamics Zehui Deng Physics Department, Beijing Normal University, Beijing 10 0875, China ...


## 3. Embed — map each chunk to $\mathbb{R}^{384}$

We use [`all-MiniLM-L6-v2`](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) — a 22M-parameter sentence encoder trained with contrastive loss on a billion sentence pairs. Inputs up to 256 tokens, outputs a 384-dimensional float32 vector. On a laptop CPU it runs around 500 chunks/second.

> **First-run cost.** The model weights (90 MB) download from HuggingFace the first time you construct `SentenceTransformer(...)`. Expect a 10–20 s one-off stall. They're cached under `~/.cache/huggingface/` after that.

The training objective makes the geometry meaningful: paraphrases are close in **cosine distance**, not Euclidean distance, which is why we configure ChromaDB to use cosine space in the next cell.


In [36]:
from sentence_transformers import SentenceTransformer

# One-off cost ~15s first time (downloads ~90 MB). Cached afterwards.
encoder = SentenceTransformer("all-MiniLM-L6-v2")
print(f"Embedding dim: {encoder.get_sentence_embedding_dimension()}")

# Encode all chunks at once. show_progress_bar=True is useful for big corpora.
texts = [c["text"] for c in all_chunks]
embeddings = encoder.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=False,  # ChromaDB handles normalisation with cosine space
)
print(f"Encoded {len(embeddings)} chunks -> shape {embeddings.shape}, dtype {embeddings.dtype}")


Embedding dim: 384


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Encoded 97 chunks -> shape (97, 384), dtype float32


## 4. Store — ChromaDB with cosine-similarity space

[ChromaDB](https://docs.trychroma.com) is a minimal, embedded vector database. For a corpus of (10^2) chunks (like our) it's total overkill. We could do brute-force cosine similarity with a single `numpy` dot product. But the ChromaDB API is the one you'll meet in production, it's persistent across notebook restarts, and the filter syntax will become useful in future lectures when we want to restrict retrieval to specific sub-corpora.

Two non-obvious choices:

- **`hnsw:space="cosine"`**: the default is `l2` (Euclidean). MiniLM embeddings are only meaningful under cosine distance, so we must override.
- **`PersistentClient(path=...)`**: writes a SQLite + HNSW index under `chroma_db/`. Later sessions can `get_collection(...)` without re-embedding.


In [37]:
import chromadb

CHROMA_DIR = Path("../chroma_db")
COLLECTION_NAME = "ising_papers"

client = chromadb.PersistentClient(path=str(CHROMA_DIR))

# Recreate from scratch so the notebook is idempotent across re-runs.
try:
    client.delete_collection(COLLECTION_NAME)
except Exception:
    pass
collection = client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)

# ChromaDB wants lists, not numpy arrays, for embeddings and metadatas.
collection.add(
    ids=[c["doc_id"] for c in all_chunks],
    documents=texts,
    embeddings=embeddings.tolist(),
    metadatas=[{
        "arxiv_id":   c["arxiv_id"],
        "source":     c["source"],
        "title":      c["title"],
        "page_start": c["page_start"],
    } for c in all_chunks],
)
print(f"Indexed {collection.count()} chunks into {CHROMA_DIR.resolve()}")


Indexed 97 chunks into /Users/Yak52/coding-lab/ising-agents-course/chroma_db


## 5. Retrieve — nearest neighbours in embedding space

Given a free-text query, we:

1. embed the query with the same `encoder` (this is critiqal: the query and corpus must live in the same space),
2. ask ChromaDB for the top-$K$ chunks,
3. get back ids, documents, distances, and metadata.

ChromaDB returns cosine *distance* $d = 1 - \cos\theta$, so smaller is better. We'll sanity-check by throwing a hand-written query at it.


In [38]:
def retrieve(query: str, top_k: int = 3) -> list[dict]:
    """Return the top-k chunks most relevant to `query`, sorted by cosine distance."""
    q_emb = encoder.encode([query], normalize_embeddings=False).tolist()
    res = collection.query(query_embeddings=q_emb, n_results=top_k)
    # ChromaDB nests results one level deep (one list per query).
    hits: list[dict] = []
    for i in range(len(res["ids"][0])):
        hits.append({
            "doc_id":   res["ids"][0][i],
            "distance": float(res["distances"][0][i]),
            "text":     res["documents"][0][i],
            "meta":     res["metadatas"][0][i],
        })
    return hits


# Sanity check: what does our corpus say about the Wolff algorithm?
demo_hits = retrieve("How does the Wolff cluster algorithm work?", top_k=3)
for h in demo_hits:
    m = h["meta"]
    print(f"[dist={h['distance']:.3f}]  {m['title'][:60]}  (p.{m['page_start']})")
    print(f"  {h['text'][:220]}...\n")


[dist=0.485]  Local and cluster critical dynamics of the 3d random-site Is  (p.9)
  above, local and cluster MC algorithms give ris e to diﬀerent forms of critical dynamics. Therefore, they are describ ed by diﬀerent scaling exponents (14), (15). Indeed, the large clusters of equa lly oriented spins are...

[dist=0.508]  Static and dynamic critical behaviour of 3d random site Isin  (p.2)
  using different algorithms. In the present analy sis, we made use of three different algorithms, Metropolis, Swendsen-Wang and Wolff ones , which enabled us to find out regions of their applicability and to discriminat e...

[dist=0.514]  Monte Carlo Simulation of Anisotropic Ising Model Using Metr  (p.3)
  white dots here represent the +1 and -1 states of the model. The first row displays the results for the Wolff algorithm, while the second row shows the lattice of the Metropolis algorithm. These snapshots were captured a...



## 6. Generate — inject retrieved context into the prompt

A "RAG answer" is just the naked LLM, with the retrieved passages prepended to the user message under a big "use only these sources" instruction. The magic is not in the LLM; it's in the fact that the passages are real.

We reuse the `chat()` primitive from notebook 01.


In [39]:
import requests

OLLAMA_URL = "http://localhost:11434/api/chat"
MODEL      =  "qwen2.5:7b" # swap to mistral:7b or "llama3.1:8b" etc. if you prefer


def chat(messages: list[dict], model: str = MODEL, temperature: float = 0.1,
         url: str = OLLAMA_URL, timeout: int = 180) -> str:
    payload = {"model": model, "messages": messages,
               "stream": False, "options": {"temperature": temperature}}
    r = requests.post(url, json=payload, timeout=timeout)
    r.raise_for_status()
    return r.json()["message"]["content"]


def rag_answer(question: str, top_k: int = 4) -> str:
    hits = retrieve(question, top_k=top_k)
    # Build a numbered bibliography so the model can cite by [1], [2], ...
    context_block = "\n\n".join(
        f"[{i+1}] {h['meta']['title']} (arXiv:{h['meta']['arxiv_id']}, p.{h['meta']['page_start']}):\n"
        f"{h['text']}"
        for i, h in enumerate(hits)
    )
    system = (
        "You are a careful scientific assistant. Answer the user\'s question using ONLY the "
        "passages provided in <context>. If the passages do not contain the answer, say so. "
        "Cite sources inline with bracketed numbers like [1] that match the numbered passages."
    )
    user = f"<context>\n{context_block}\n</context>\n\nQuestion: {question}"
    reply = chat([{"role": "system", "content": system},
                  {"role": "user",   "content": user}])
    # Return both for inspection
    return reply, hits


answer, hits = rag_answer("What is the Wolff algorithm and how does it differ from Metropolis?")
print(answer)
print("\n--- sources ---")
for i, h in enumerate(hits):
    print(f"[{i+1}] {h['meta']['title'][:70]} (p.{h['meta']['page_start']})  dist={h['distance']:.3f}")


The Wolff algorithm is a cluster-based Monte Carlo method used to simulate lattice models, particularly useful in the vicinity of phase transitions [3]. It differs from the Metropolis algorithm in several key ways:

1. **Cluster Flipping**: Instead of flipping individual spins as in the Metropolis algorithm, the Wolff algorithm flips entire clusters of connected spins simultaneously [2].

2. **Critical Slowing Down Mitigation**: Near critical points where correlation lengths diverge, single-spin flip methods like Metropolis can suffer from critical slowing down—where autocorrelation times increase dramatically. The Wolff algorithm reduces this issue by flipping larger clusters, thus de-correlating the system more efficiently [4].

3. **Efficiency at Critical Points**: For systems near their phase transition temperatures, the Wolff algorithm is significantly faster and more efficient compared to the Metropolis method due to its ability to handle large-scale changes in the spin configura

## 7. Wrap as a `smolagents` tool + rebuild the agent

Now we expose retrieval to the Module 1.1 agent. The tool returns a compact, human-readable summary of the top passages (*not* the raw embeddings). The agent then decides whether to use it or not, possibly alongside the calculator and unit converter.

This is the real payoff. The agent was already good at arithmetic; now it can also answer *bibliographic* questions that would otherwise require the model to invent references.


In [46]:
from smolagents import ToolCallingAgent, CodeAgent, LiteLLMModel, tool


@tool
def search_papers(query: str, top_k: int = 3) -> str:
    """Search the local arXiv corpus (Ising-model papers) for passages relevant to a query.

    Returns a newline-separated list of snippets with citations. Use this tool when
    the question is about what specific papers say: methods, results, claims,
    numerical values, historical attribution.

    Args:
        query: a natural-language question or search phrase.
        top_k: how many passages to return (1-5). Default 3.
    """
    hits = retrieve(query, top_k=max(1, min(top_k, 5)))
    lines = []
    for i, h in enumerate(hits):
        m = h["meta"]
        snippet = h["text"][:500].replace("\n", " ")
        lines.append(
            f"[{i+1}] {m['title']} (arXiv:{m['arxiv_id']}, p.{m['page_start']}): {snippet}..."
        )
    return "\n\n".join(lines) if lines else "(no relevant passages found)"


# Re-import the tools we built in notebook 02. For the course they live inline;
# outside the course they'd sit in a `tools/` module.
import ast, math, operator as op

_ALLOWED_BINOPS = {ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul,
                   ast.Div: op.truediv, ast.Pow: op.pow, ast.Mod: op.mod, ast.FloorDiv: op.floordiv}
_ALLOWED_UNARYOPS = {ast.UAdd: op.pos, ast.USub: op.neg}
_ALLOWED_NAMES = {"pi": math.pi, "e": math.e, "sqrt": math.sqrt, "log": math.log,
                  "log10": math.log10, "exp": math.exp, "sin": math.sin,
                  "cos": math.cos, "tan": math.tan, "abs": abs}


def _safe_eval(node):
    if isinstance(node, ast.Expression):         return _safe_eval(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return float(node.value)
    if isinstance(node, ast.Name) and node.id in _ALLOWED_NAMES:
        v = _ALLOWED_NAMES[node.id]
        return v if callable(v) else float(v)
    if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_BINOPS:
        return _ALLOWED_BINOPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _ALLOWED_UNARYOPS:
        return _ALLOWED_UNARYOPS[type(node.op)](_safe_eval(node.operand))
    if isinstance(node, ast.Call) and isinstance(node.func, ast.Name):
        fn = _ALLOWED_NAMES.get(node.func.id)
        if callable(fn) and not node.keywords:
            return fn(*[_safe_eval(a) for a in node.args])
    raise ValueError(f"disallowed expression: {ast.dump(node)}")


@tool
def calculator_tool(expression: str) -> float:
    """Evaluate a math expression (sqrt, log, exp, trig, pi, e).

    Args:
        expression: a math expression, e.g. "2*pi*sqrt(3)".
    """
    return float(_safe_eval(ast.parse(expression, mode="eval")))


model = LiteLLMModel(
    model_id=f"ollama_chat/{MODEL}",
    api_base="http://localhost:11434",
    api_key="ollama",
    temperature=0.0,
)

#ToolCallingAgent (only with big models): it requires the model to produce a specific JSON schema with function name, 
# arguments, and a separate final_answer tool call — all formatted exactly right. 
# Small models frequently break this schema.

#agent = ToolCallingAgent(
#    tools=[search_papers, calculator_tool],
#    model=model,
#    max_steps=6,
#    #planning_interval=1,  # forces the model to plan before each step
#)

# CodeAgent (works with all models): just asks the model to write Python.
#  The model already knows Python well, so print(search_papers(...)) and final_answer(...) are natural to produce. 
# The barrier to entry is much lower.

agent = CodeAgent(
    tools=[search_papers, calculator_tool],
    model=model,
    max_steps=6,
)

print("Agent built with tools:", [t.name for t in agent.tools.values()])


Agent built with tools: ['search_papers', 'calculator_tool', 'final_answer']


## 8. Three physics questions — with citations

These are the tasks from the course outline. Watch the agent decide whether to call `search_papers` or not. A well-behaved run will show one or two tool calls, then a cited answer.

> If your model struggles (empty args, loops, or refusing to cite), scroll back to §5 and read the raw retrieval results — that tells you whether the problem is *retrieval* (garbage in) or *generation* (model misbehaving).


In [47]:
q1 = ("What numerical methods have been used to estimate critical exponents in "
      "the 2D or 3D Ising model according to the papers in our corpus? Cite sources.")
print(agent.run(q1))


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What numerical methods have been used to estimate critical exponents in the 2D or 3D Ising model according to   │
│ the papers in our corpus? Cite sources.                                                                         │
│                                                                                                                 │
╰─ LiteLLMModel - ollama_chat/qwen2.5:7b ─────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  passages = search_papers(query="numerical methods estimating critical exponents 2D 3D Ising model", top_k=5)     
  print(passages)                                                                                                  
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
[1] Local and cluster critical dynamics of the 3d random-site Ising model (arXiv:cond-mat_0603521, p.20): α < βfor 
the 3d Ising model, comparison of Eqs. (20) and (21) leads to the inequality: zW E,int< zSW E,int. (22) Eq. (21) 
does not hold for the diluted systems (where the heat cap acity crit- ical exponent is negative). However, the 
inequality (22) still holds, as one can see, comparing data of Table 6 for the Wolﬀ and Swendsen-Wang algo- rithms.
Again, the value of dynamical critical exponent for the dilut ed system is smaller than its counterpart for the 
pure one. Wolﬀ algorithm dyna mical cri...

[2] Static and dynamic critical behaviour of 3d random site Ising model: different Monte Carlo algorithms 
(arXiv:cond-mat_0508719, p.1): arXiv:cond-mat/0508719 30 Aug 2005 1 Static and dynamic critical behaviour of 3d 
random site Ising model: different Monte Carlo algorithms D. Ivaneyko a, J. Ilnytskyi b, B. Berche c,d , and Yu. 
Holovatch b,c,a a Ivan Franko National University of Lviv, 79005 Lviv, Ukaine b Institute for Condensed Matter 
Physics, National Academy of Sciences of Ukraine, 79011 Lviv, Ukraine c Laboratoire de Physique des Matériaux, 
Université Henri Poincaré, Nancy 1, 54506 Vandœuvre les Nancy Cedex, France d Instit...

[3] Local and cluster critical dynamics of the 3d random-site Ising model (arXiv:cond-mat_0603521, p.1): 
arXiv:cond-mat/0603521v1 [cond-mat.dis-nn] 20 Mar 2006Local and cluster critical dynamics of the 3d random-site 
Ising model D. Ivaneykoa,∗, J. Ilnytskyib, B. Berchec, Yu. Holovatchb,d,a aIvan Franko National University of Lviv,
79005 Lviv, Ukraine bInstitute for Condensed Matter Physics, National Acad. Sci. o f Ukraine, 79011 Lviv, Ukraine 
cLaboratoire de Physique des Mat´ eriaux, Universit´ e Henri Poincar´ e, Nancy 1, 54506 Vandœuvre les Nancy Cedex, 
France dInstitut f¨ ur Theoretische Physik,...

[4] Two temperature Ising Model (arXiv:2008.00245, p.1): Two temperature Ising Model J. Cheraghalizadeh*,1M. 
Seiﬁ,1Z. Ebadi,1H. Mohammadzadeh,1and M. N. Najaﬁ1 1Department of Physics, University of Mohaghegh Ardabili, P.O. 
Box 179, Ardabil, Iran∗ We introduce a two-temperature Ising model as a prototype of superstatistic critical 
phenomena. The model is described by two temperatures ( T1,T2) in zero magnetic ﬁeld. To predict the phase diagram 
and numerically estimate the exponents, we develop Metropolis and Swendsen-Wang Monte Carlo method. We observe...

[5] Monte Carlo Simulation of Anisotropic Ising Model Using Metropolis and Wolff Algorithm (arXiv:2411.06819, p.1):
for J x,Jy<0, the material becomes antiferromagnetic. When J = 0, the spins do not interact with each other. In 
these simulations, Jis set as the unit of energy and temperature. The changes in the value of J is explained ahead 
for different anisotropic scenarios.arXiv:2411.06819v1 [cond-mat.stat-mech] 11 Nov 2024 2 By the term anisotropic 
Ising model, we mean thatJy Jx̸= 1.0. This is the model we are interested in our simulations. . FIG. 1. Schematic 
diagram of 2D Ising model with JxandJyas near...

Out: None

[Step 1: Duration 20.99 seconds| Input tokens: 2,239 | Output tokens: 81]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("The numerical methods used to estimate critical exponents in the 2D or 3D Ising model according    
  to the papers in our corpus are Monte Carlo algorithms such as the Metropolis algorithm and the Swendsen-Wang    
  algorithm. Citations: arXiv:cond-mat_0603521, arXiv:cond-mat_0508719")                                           
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Out - Final answer: The numerical methods used to estimate critical exponents in the 2D or 3D Ising model according
to the papers in our corpus are Monte Carlo algorithms such as the Metropolis algorithm and the Swendsen-Wang 
algorithm. Citations: arXiv:cond-mat_0603521, arXiv:cond-mat_0508719

[Step 2: Duration 14.55 seconds| Input tokens: 5,636 | Output tokens: 231]

The numerical methods used to estimate critical exponents in the 2D or 3D Ising model according to the papers in our corpus are Monte Carlo algorithms such as the Metropolis algorithm and the Swendsen-Wang algorithm. Citations: arXiv:cond-mat_0603521, arXiv:cond-mat_0508719


In [42]:
q2 = ("According to our corpus, what is the Wolff cluster algorithm, and how "
      "does it compare to the Metropolis algorithm for simulating the Ising model "
      "near T_c? Cite specific papers.")
print(agent.run(q2))


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ According to our corpus, what is the Wolff cluster algorithm, and how does it compare to the Metropolis         │
│ algorithm for simulating the Ising model near T_c? Cite specific papers.                                        │
│                                                                                                                 │
╰─ LiteLLMModel - ollama_chat/qwen2.5:7b ─────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  query = "Wolff cluster algorithm compared to Metropolis algorithm for Ising model near T_c"                      
  papers = search_papers(query=query, top_k=3)                                                                     
  print(papers)                                                                                                    
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
[1] Monte Carlo Simulation of Anisotropic Ising Model Using Metropolis and Wolff Algorithm (arXiv:2411.06819, p.5):
fields in section V. Thereafter, we are presenting the hysteresis loop survey for the Ising model for different 
Isotropic cases in section VI. Finally, we go to the magnetocaloric effect in section VII. And the conclusion of 
the paper in the last section VIII. II. THE METROPOLIS AND WOLFF ALGORITHM The Metropolis and the Wolff algorithm 
have been well studied in the literature for example in [36–40]. Here we are giving the overview of the 
step-by-step process of how these algorithms work [21, 32...

[2] Monte Carlo Simulation of Anisotropic Ising Model Using Metropolis and Wolff Algorithm (arXiv:2411.06819, p.3):
white dots here represent the +1 and -1 states of the model. The first row displays the results for the Wolff 
algorithm, while the second row shows the lattice of the Metropolis algorithm. These snapshots were captured after 
103Monte Carlo steps for 100 ×100 lattice. For temperatures near the critical temperature, the Wolff algorithm 
reaches equilibrium near 103steps, whereas the Metropolis algorithm lags. A video demonstrating these flips has 
been created and an online video can be seen by clic...

[3] Monte Carlo Simulation of Anisotropic Ising Model Using Metropolis and Wolff Algorithm (arXiv:2411.06819, p.1):
for J x,Jy<0, the material becomes antiferromagnetic. When J = 0, the spins do not interact with each other. In 
these simulations, Jis set as the unit of energy and temperature. The changes in the value of J is explained ahead 
for different anisotropic scenarios.arXiv:2411.06819v1 [cond-mat.stat-mech] 11 Nov 2024 2 By the term anisotropic 
Ising model, we mean thatJy Jx̸= 1.0. This is the model we are interested in our simulations. . FIG. 1. Schematic 
diagram of 2D Ising model with JxandJyas near...

Out: None

[Step 1: Duration 10.43 seconds| Input tokens: 2,242 | Output tokens: 91]

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("The Wolff cluster algorithm is more efficient for simulating the Ising model near the critical     
  temperature \(T_c\) compared to the Metropolis algorithm, as it reaches equilibrium faster.")                    
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Out - Final answer: The Wolff cluster algorithm is more efficient for simulating the Ising model near the critical 
temperature \(T_c\) compared to the Metropolis algorithm, as it reaches equilibrium faster.

[Step 3: Duration 10.88 seconds| Input tokens: 8,738 | Output tokens: 469]

The Wolff cluster algorithm is more efficient for simulating the Ising model near the critical temperature \(T_c\) compared to the Metropolis algorithm, as it reaches equilibrium faster.


In [43]:
q3 = ("Do any papers in our corpus discuss the Ising model on non-regular or "
      "non-square lattices (e.g. Barabasi-Albert networks, random sites)? If so, "
      "what did they find?")
print(agent.run(q3))


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Do any papers in our corpus discuss the Ising model on non-regular or non-square lattices (e.g. Barabasi-Albert │
│ networks, random sites)? If so, what did they find?                                                             │
│                                                                                                                 │
╰─ LiteLLMModel - ollama_chat/qwen2.5:7b ─────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  query = "Ising model non-regular lattice Barabasi-Albert network random sites"                                   
  search_results = search_papers(query=query, top_k=3)                                                             
  print(search_results)                                                                                            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
[1] Ising model spin S=1 on directed Barabasi-Albert networks (arXiv:cond-mat_0607220, p.1): e scale-free networks,
diﬀerent algorithms give diﬀerent results. Now we study the Ising model for spin S= 1 on directed Barab´ asi-Albert
network and diﬀerent from the Ising model for spin S= 1/2, the order-disorder phase transition of or- der parameter
is well deﬁned in this system. We have obtained a ﬁrst -order 1 phase transition for values of connectivity m= 2 
and m= 7 of the directed Barab´ asi-Albert network. 0.0 1.0 2.0 3.0 4.0 
temperature0.00.20.40.60.81.0magnetisationspin−1, m=2 N=250 ...

[2] Local and cluster critical dynamics of the 3d random-site Ising model (arXiv:cond-mat_0603521, p.10): vast 
amount of single spin ﬂips are rejected, therefore the conﬁgurations gen erated are highly correlated and the 
system moves in a phase space ineﬃciently. Cluster algorithmswere designed toovercome thesediﬃculties. The 
yarebased on the identiﬁcation of clusters of sites using a bond percolation pro cess con- nected to the spin 
conﬁguration of the magnetic system. All spins o f the clus- ters are then independently ﬂipped. In the case of the
Ising model (or more generally the Potts model), the...

[3] Mixed algorithms in the Ising model on directed Barabasi-Albert networks (arXiv:cond-mat_0508666, p.1): Barab´ 
asi-Albert networks. They found a freezing-in of the magn etisation 1 similar to [1, 2], following an Arrhenius law
at least in low dimensions. This lack of a spontaneous magnetisation (in the usual sense) is consist ent with the 
fact that if on a directed lattice a spin Sjinﬂuences spin Si, then spin Si in turn does not inﬂuence Sj, and there
may be no well-deﬁned total energy. Thus, they show that for the same scale-free networks, diﬀeren t algorithms 
give diﬀerent results. Now we study...

Out: None

[Step 1: Duration 9.92 seconds| Input tokens: 2,247 | Output tokens: 83]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("The papers found discuss the Ising model on non-regular or non-square lattices such as             
  Barabasi-Albert networks and random sites. They find that different algorithms give different results, a         
  first-order phase transition is observed for certain connectivity values in directed Barabasi-Albert networks,   
  single spin flips are rejected due to highly correlated configurations generated by the system moving            
  inefficiently through phase space, and there can be freezing-in of magnetization following an Arrhenius law at   
  least in low dimensions.")                                                                                       
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Out - Final answer: The papers found discuss the Ising model on non-regular or non-square lattices such as 
Barabasi-Albert networks and random sites. They find that different algorithms give different results, a 
first-order phase transition is observed for certain connectivity values in directed Barabasi-Albert networks, 
single spin flips are rejected due to highly correlated configurations generated by the system moving inefficiently
through phase space, and there can be freezing-in of magnetization following an Arrhenius law at least in low 
dimensions.

[Step 2: Duration 20.81 seconds| Input tokens: 5,219 | Output tokens: 399]

The papers found discuss the Ising model on non-regular or non-square lattices such as Barabasi-Albert networks and random sites. They find that different algorithms give different results, a first-order phase transition is observed for certain connectivity values in directed Barabasi-Albert networks, single spin flips are rejected due to highly correlated configurations generated by the system moving inefficiently through phase space, and there can be freezing-in of magnetization following an Arrhenius law at least in low dimensions.


## 9. Key takeaway + exercises

**The shape of the pipeline.** RAG is six steps, all of them ordinary: read files, split text, encode with a pre-trained model, index vectors, query by nearest neighbour, paste results into a prompt. No step requires fine-tuning, no step is "AI magic." The magic is that a 22M-parameter sentence encoder has already learned a useful geometry on a billion pairs of sentences.

**What RAG buys you.** Facts, citations, and a knob (`top_k`) to trade precision vs recall. What it doesn't buy you: reasoning. If the answer requires synthesising across three papers, the LLM still does that synthesis and can still get it wrong. That's why the next module (1.3) adds *action* tools (run a simulation via MCP), and Module 1.4 adds *reflection* (the agent critiques its own output).

### Exercises

1. **Retrieval quality audit.** Run `retrieve("critical exponents beta gamma nu", top_k=10)` and eyeball the distances. Where does the signal die? Try `top_k=20`. Is there a clear elbow in the distance curve, or is it smooth? (If smooth, your corpus is probably too narrow — all the papers are Ising papers, so almost everything is "somewhat relevant".)

2. **PDF extraction failure.** Pick one paper and compare `papers[i]['pages'][3]` to the actual PDF page 3. What got mangled? (Look for equations and two-column layouts.) Propose one concrete fix — a different library, a cleanup regex, or switching to OCR.

3. **Chunk-size sweep.** Re-run §2–§4 with `(size=200, overlap=25)` and again with `(size=800, overlap=100)`. Ask the *same* question through `rag_answer` in all three settings. Which gives the most specific, well-cited answer? There is no universal winner — the answer tells you something about your corpus and your questions.

4. **Negative retrieval.** Ask the agent something your corpus can't answer, e.g. *"What is the exact T_c value for the 2D Heisenberg model?"* (Not in our Ising corpus.) Does the agent correctly say "not in the corpus"? Or does it hallucinate? This is the failure mode Module 1.4 (reflection) is designed to catch.

---

**Next up — Module 1.3:** We give the agent *action* tools via the **Model Context Protocol** (MCP). The agent will be able to `run_ising_simulation(L=32, T=2.269)` and get real numerical data back not from a library, but from a live Monte Carlo run. The RAG pipeline you built here stays in place; MCP adds a new *kind* of tool alongside it.
